In [ ]:
import pandas as pd
import numpy as np

# 1. File paths

voyage_path = "compiled_voyages_2018_2021_tyndall_v1.parquet"

vessel_path = "Kpler_vessel_particulars_2024_csv.csv"

port_path = "port_location_matched_s2id.csv"

output_path = "voyages_enriched.parquet"

In [3]:
# 2. Load source datasets

# Load the voyage-level emissions dataset.
voyages = pd.read_parquet(voyage_path)

# Load selected vessel attributes.
vessels = pd.read_csv(
    vessel_path,
    usecols=[
        "imo",
        "mmsi",
        "name",
        "commercialFleet",
        "generalVesselType",
        "detailedVesselType",
        "serviceStatus",
        "flag",
        "grossTonnage",
        "deadweightTonnage",
        "enginePower",
        "yearOfBuild",
        "mainEngineFuelType",
        "beneficialOwner_country",
        "registeredOwner_country",
        "operator_country",
        "technicalManager_country",
    ],
)

# Load selected port attributes.
ports = pd.read_csv(
    port_path,
    usecols=[
        "s2id",
        "lat",
        "lon",
        "label",
        "sublabel",
        "iso3",
        "distance_from_shore_m",
        "dock",
        "PortID",
        "closest_lat",
        "closest_lon",
    ],
    dtype={
        "s2id": "string",
        "label": "string",
        "sublabel": "string",
        "iso3": "string",
        "PortID": "string",
    },
    low_memory=False,
)

In [4]:
# 3. Standardise join keys

# IMO is the main vessel identifier.
voyages["imo"] = pd.to_numeric(voyages["imo"], errors="coerce")
vessels["imo"] = pd.to_numeric(vessels["imo"], errors="coerce")

# Port IDs in the voyage data match the s2id field in the port table.
voyages["port_id_o"] = voyages["port_id_o"].astype(str)
voyages["port_id_d"] = voyages["port_id_d"].astype(str)
ports["s2id"] = ports["s2id"].astype(str)

In [5]:
# 4. Remove duplicate keys

# The dimension tables should have one row per join key.
vessels = vessels.drop_duplicates(subset=["imo"])
ports = ports.drop_duplicates(subset=["s2id"])

In [6]:
# 5. Check match rates before merging

voyage_imos = set(voyages["imo"].dropna().astype("int64").unique())
vessel_imos = set(vessels["imo"].dropna().astype("int64").unique())

all_voyage_ports = pd.concat(
    [
        voyages["port_id_o"],
        voyages["port_id_d"],
    ],
    ignore_index=True,
)

voyage_port_ids = set(all_voyage_ports.dropna().astype(str).unique())
port_s2ids = set(ports["s2id"].dropna().astype(str).unique())

imo_match_rate = len(voyage_imos & vessel_imos) / len(voyage_imos)
port_match_rate = len(voyage_port_ids & port_s2ids) / len(voyage_port_ids)

print("Voyage rows:", len(voyages))
print("Vessel rows:", len(vessels))
print("Port rows:", len(ports))
print(f"Unique voyage IMO count: {len(voyage_imos):,}")
print(f"Matched IMO count: {len(voyage_imos & vessel_imos):,}")
print(f"IMO match rate: {imo_match_rate:.2%}")
print(f"Unique voyage port ID count: {len(voyage_port_ids):,}")
print(f"Matched port ID count: {len(voyage_port_ids & port_s2ids):,}")
print(f"Port match rate: {port_match_rate:.2%}")

Voyage rows: 5838724
Vessel rows: 227100
Port rows: 166482
Unique voyage IMO count: 25,884
Matched IMO count: 24,252
IMO match rate: 93.69%
Unique voyage port ID count: 16,290
Matched port ID count: 16,290
Port match rate: 100.00%


In [7]:
# 6. Prepare dimension tables

# Prefix vessel fields to avoid confusion with voyage-level fields.
vessels_for_merge = vessels.rename(
    columns={
        "mmsi": "vessel_mmsi",
        "name": "vessel_name",
        "commercialFleet": "vessel_commercial_fleet",
        "generalVesselType": "vessel_general_type",
        "detailedVesselType": "vessel_detailed_type",
        "serviceStatus": "vessel_service_status_2024",
        "flag": "vessel_flag_2024",
        "grossTonnage": "vessel_gross_tonnage",
        "deadweightTonnage": "vessel_deadweight_tonnage",
        "enginePower": "vessel_engine_power",
        "yearOfBuild": "vessel_year_of_build",
        "mainEngineFuelType": "vessel_main_engine_fuel_type",
        "beneficialOwner_country": "vessel_beneficial_owner_country_2024",
        "registeredOwner_country": "vessel_registered_owner_country_2024",
        "operator_country": "vessel_operator_country_2024",
        "technicalManager_country": "vessel_technical_manager_country_2024",
    }
)

origin_ports = ports.rename(
    columns={
        "s2id": "origin_s2id",
        "lat": "origin_lat",
        "lon": "origin_lon",
        "label": "origin_port_label",
        "sublabel": "origin_port_sublabel",
        "iso3": "origin_port_iso3",
        "distance_from_shore_m": "origin_distance_from_shore_m",
        "dock": "origin_dock",
        "PortID": "origin_port_id_reference",
        "closest_lat": "origin_closest_lat",
        "closest_lon": "origin_closest_lon",
    }
)

destination_ports = ports.rename(
    columns={
        "s2id": "destination_s2id",
        "lat": "destination_lat",
        "lon": "destination_lon",
        "label": "destination_port_label",
        "sublabel": "destination_port_sublabel",
        "iso3": "destination_port_iso3",
        "distance_from_shore_m": "destination_distance_from_shore_m",
        "dock": "destination_dock",
        "PortID": "destination_port_id_reference",
        "closest_lat": "destination_closest_lat",
        "closest_lon": "destination_closest_lon",
    }
)

In [8]:
# 7. Merge datasets

# Merge vessel particulars onto voyage records.
df = voyages.merge(
    vessels_for_merge,
    on="imo",
    how="left",
    validate="many_to_one",
)

# Merge origin port information.
df = df.merge(
    origin_ports,
    left_on="port_id_o",
    right_on="origin_s2id",
    how="left",
    validate="many_to_one",
)

# Merge destination port information.
df = df.merge(
    destination_ports,
    left_on="port_id_d",
    right_on="destination_s2id",
    how="left",
    validate="many_to_one",
)

In [9]:
# 8. Create derived time fields

df["departure_time"] = pd.to_datetime(
    df["unixtimestamp_o"],
    unit="s",
    errors="coerce",
)

df["arrival_time"] = pd.to_datetime(
    df["unixtimestamp_d"],
    unit="s",
    errors="coerce",
)

df["departure_year"] = df["departure_time"].dt.year
df["departure_month"] = df["departure_time"].dt.to_period("M").astype(str)
df["departure_quarter"] = df["departure_time"].dt.to_period("Q").astype(str)

df["arrival_year"] = df["arrival_time"].dt.year
df["arrival_month"] = df["arrival_time"].dt.to_period("M").astype(str)

In [14]:
# 9. Create route fields

df["country_o"] = df["country_o"].fillna("Unknown")
df["country_d"] = df["country_d"].fillna("Unknown")

df["voyage_scope"] = np.where(
    df["country_o"].eq(df["country_d"]),
    "Domestic",
    "International",
)

df["route_directional"] = (
    df["country_o"].astype(str)
    + " → "
    + df["country_d"].astype(str)
)

pair_left = df[["country_o", "country_d"]].astype(str).min(axis=1)
pair_right = df[["country_o", "country_d"]].astype(str).max(axis=1)

df["country_pair"] = pair_left + " ↔ " + pair_right

df["port_route_directional"] = (
    df["origin_port_label"].fillna(df["port_id_o"]).astype(str)
    + " → "
    + df["destination_port_label"].fillna(df["port_id_d"]).astype(str)
)

df["origin_port_display"] = df["origin_port_sublabel"].fillna(df["origin_port_label"]).fillna(df["port_id_o"])

df["destination_port_display"] = df["destination_port_sublabel"].fillna(df["destination_port_label"]).fillna(df["port_id_d"])

df["port_route_display"] = (
    df["origin_port_display"].astype(str)
    + " → "
    + df["destination_port_display"].astype(str)
)

In [15]:
# 10. Create vessel fields

df["ship_type_clean"] = df["astd_ship_type"].fillna("Unknown")

df["vessel_age_at_departure"] = np.where(
    df["departure_year"].notna() & df["vessel_year_of_build"].notna(),
    df["departure_year"] - df["vessel_year_of_build"],
    np.nan,
)

df["vessel_age_group"] = pd.cut(
    df["vessel_age_at_departure"],
    bins=[-np.inf, 5, 10, 20, 30, np.inf],
    labels=[
        "0-5 years",
        "6-10 years",
        "11-20 years",
        "21-30 years",
        "30+ years",
    ],
)

df["vessel_size_group_dwt"] = pd.cut(
    df["dwt"],
    bins=[-np.inf, 10_000, 50_000, 100_000, 200_000, np.inf],
    labels=[
        "<10k DWT",
        "10k-50k DWT",
        "50k-100k DWT",
        "100k-200k DWT",
        "200k+ DWT",
    ],
)

In [16]:
# 11. Create emission intensity fields

df["CO2_per_km"] = np.where(
    df["delta_dist_km"] > 0,
    df["CO2"] / df["delta_dist_km"],
    np.nan,
)

df["CO2_per_dwt_km"] = np.where(
    (df["delta_dist_km"] > 0) & (df["dwt"] > 0),
    df["CO2"] / (df["delta_dist_km"] * df["dwt"]),
    np.nan,
)

df["ME_CO2_share"] = np.where(
    df["CO2"] > 0,
    df["CO2_ME_kg"] / df["CO2"],
    np.nan,
)

df["AE_CO2_share"] = np.where(
    df["CO2"] > 0,
    df["CO2_AE_kg"] / df["CO2"],
    np.nan,
)

df["boiler_CO2_share"] = np.where(
    df["CO2"] > 0,
    df["CO2_boiler_kg"] / df["CO2"],
    np.nan,
)


In [18]:
# 12. Final validation checks

print("\nAfter merge:")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nDate range:")
print("Departure:", df["departure_time"].min(), "to", df["departure_time"].max())
print("Arrival:", df["arrival_time"].min(), "to", df["arrival_time"].max())

print("\nPost-merge match rates:")
print(f"Vessel name available: {df['vessel_name'].notna().mean():.2%}")
print(f"Origin port label available: {df['origin_port_label'].notna().mean():.2%}")
print(f"Destination port label available: {df['destination_port_label'].notna().mean():.2%}")

print("\nSample enriched rows:")
sample_cols = [
    "imo",
    "vessel_name",
    "country_o",
    "origin_port_label",
    "origin_port_sublabel",
    "origin_port_display",
    "country_d",
    "destination_port_label",
    "destination_port_sublabel",
    "destination_port_display",
    "port_route_display",
    "ship_type_clean",
    "vessel_general_type",
    "vessel_flag_2024",
    "vessel_year_of_build",
    "vessel_age_at_departure",
    "CO2",
    "CO2_per_km",
]

print(df[sample_cols].head(10).to_string(index=False))


After merge:
Rows: 5838724
Columns: 82

Date range:
Departure: 2017-01-02 21:19:13 to 2021-12-31 23:48:38
Arrival: 2018-01-01 00:15:28 to 2022-12-30 13:35:10

Post-merge match rates:
Vessel name available: 96.03%
Origin port label available: 100.00%
Destination port label available: 100.00%

Sample enriched rows:
    imo   vessel_name country_o origin_port_label origin_port_sublabel origin_port_display country_d destination_port_label destination_port_sublabel destination_port_display port_route_display ship_type_clean vessel_general_type vessel_flag_2024  vessel_year_of_build  vessel_age_at_departure      CO2  CO2_per_km
9200419 FEDERAL ASAHI       CAN             SOREL                SOREL               SOREL       FRA                  ROUEN                     ROUEN                    ROUEN      SOREL → ROUEN   Bulk carriers        BULK CARRIER      MARSHALL IS                2000.0                     20.0  49222.6  105.401713
9858424       STARNES       NOR             JELSA     

In [19]:
print("Origin display available:", df["origin_port_display"].notna().mean())
print("Destination display available:", df["destination_port_display"].notna().mean())

print(df[
    [
        "origin_port_label",
        "origin_port_sublabel",
        "origin_port_display",
        "destination_port_label",
        "destination_port_sublabel",
        "destination_port_display",
    ]
].head(20).to_string(index=False))

Origin display available: 1.0
Destination display available: 1.0
origin_port_label origin_port_sublabel origin_port_display destination_port_label destination_port_sublabel destination_port_display
            SOREL                SOREL               SOREL                  ROUEN                     ROUEN                    ROUEN
            JELSA                JELSA               JELSA                NOR-145                        BA                       BA
            JELSA                JELSA               JELSA                  JELSA                     JELSA                    JELSA
           AARHUS               AARHUS              AARHUS                  JELSA                     JELSA                    JELSA
           AARHUS               AARHUS              AARHUS                 AARHUS                    AARHUS                   AARHUS
        MEKJARVIK            RANDABERG           RANDABERG                 AARHUS                    AARHUS                   AARHUS
    

In [20]:
df.to_parquet(output_path, index=False)

print(f"\nSaved enriched dataset to: {output_path}")


Saved enriched dataset to: voyages_enriched.parquet
